# Real-Time Camera Segmentation with YOLO11

This notebook demonstrates real-time object segmentation using your camera feed with YOLO11.
The segmentation results are displayed in real-time with options to record the output.


In [ ]:
# Import required libraries
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
import threading
from datetime import datetime
import os


In [ ]:
# Load the YOLO11 segmentation model
model = YOLO("yolo11n-seg.pt")  # Using nano model for faster real-time processing
print(f"Model loaded: {model.model_name}")
print("Model is ready for real-time segmentation!")


In [ ]:
# Camera and processing configuration
CAMERA_INDEX = 0  # Usually 0 for built-in camera, 1 for external USB camera
CONFIDENCE_THRESHOLD = 0.5
FRAME_WIDTH = 640  # Reduce for better performance
FRAME_HEIGHT = 480
FPS_TARGET = 30
RECORD_OUTPUT = False  # Set to True to record the segmented video
OUTPUT_FILENAME = f"camera_segmentation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"

print(f"Camera configuration:")
print(f"  Camera index: {CAMERA_INDEX}")
print(f"  Resolution: {FRAME_WIDTH}x{FRAME_HEIGHT}")
print(f"  Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"  Recording: {'Enabled' if RECORD_OUTPUT else 'Disabled'}")
if RECORD_OUTPUT:
    print(f"  Output file: {OUTPUT_FILENAME}")


In [ ]:
# Test camera connection
def test_camera(camera_index=0):
    """Test if camera is accessible"""
    cap = cv2.VideoCapture(camera_index)
    
    if not cap.isOpened():
        print(f"❌ Cannot access camera {camera_index}")
        print("Available camera indices to try:")
        for i in range(5):  # Test first 5 indices
            test_cap = cv2.VideoCapture(i)
            if test_cap.isOpened():
                print(f"  ✅ Camera {i} is available")
                test_cap.release()
            else:
                print(f"  ❌ Camera {i} not available")
        return False
    
    # Test frame capture
    ret, frame = cap.read()
    if ret:
        height, width = frame.shape[:2]
        print(f"✅ Camera {camera_index} is working!")
        print(f"  Native resolution: {width}x{height}")
        cap.release()
        return True
    else:
        print(f"❌ Camera {camera_index} cannot capture frames")
        cap.release()
        return False

# Test the camera
camera_available = test_camera(CAMERA_INDEX)


In [ ]:
class RealTimeSegmentation:
    def __init__(self, model, camera_index=0, conf_threshold=0.5, 
                 frame_width=640, frame_height=480, record=False, output_file=None):
        self.model = model
        self.camera_index = camera_index
        self.conf_threshold = conf_threshold
        self.frame_width = frame_width
        self.frame_height = frame_height
        self.record = record
        self.output_file = output_file
        
        # Performance tracking
        self.fps_counter = 0
        self.fps_start_time = time.time()
        self.current_fps = 0
        self.processing_times = []
        
        # Control flags
        self.running = False
        self.cap = None
        self.out = None
        
    def initialize_camera(self):
        """Initialize camera capture"""
        self.cap = cv2.VideoCapture(self.camera_index)
        
        if not self.cap.isOpened():
            raise ValueError(f"Cannot open camera {self.camera_index}")
        
        # Set camera properties
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.frame_width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.frame_height)
        self.cap.set(cv2.CAP_PROP_FPS, FPS_TARGET)
        
        # Get actual properties
        actual_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        actual_fps = self.cap.get(cv2.CAP_PROP_FPS)
        
        print(f"Camera initialized:")
        print(f"  Resolution: {actual_width}x{actual_height}")
        print(f"  FPS: {actual_fps}")
        
        # Initialize video writer if recording
        if self.record and self.output_file:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            self.out = cv2.VideoWriter(self.output_file, fourcc, 20.0, (actual_width, actual_height))
            print(f"  Recording to: {self.output_file}")
    
    def cleanup(self):
        """Clean up resources"""
        if self.cap:
            self.cap.release()
        if self.out:
            self.out.release()
        cv2.destroyAllWindows()
    
    def update_fps(self):
        """Update FPS counter"""
        self.fps_counter += 1
        if self.fps_counter % 30 == 0:  # Update every 30 frames
            current_time = time.time()
            elapsed = current_time - self.fps_start_time
            self.current_fps = 30 / elapsed
            self.fps_start_time = current_time
    
    def process_frame(self, frame):
        """Process a single frame with YOLO segmentation"""
        start_time = time.time()
        
        # Run YOLO segmentation
        results = self.model(frame, conf=self.conf_threshold, verbose=False)
        
        # Get annotated frame
        annotated_frame = results[0].plot()
        
        # Add performance info to frame
        processing_time = time.time() - start_time
        self.processing_times.append(processing_time)
        
        # Keep only last 100 processing times for average
        if len(self.processing_times) > 100:
            self.processing_times = self.processing_times[-100:]
        
        avg_processing_time = np.mean(self.processing_times)
        processing_fps = 1.0 / avg_processing_time if avg_processing_time > 0 else 0
        
        # Add text overlay with performance info
        cv2.putText(annotated_frame, f"FPS: {self.current_fps:.1f}", 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(annotated_frame, f"Processing FPS: {processing_fps:.1f}", 
                   (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(annotated_frame, f"Objects: {len(results[0].boxes) if results[0].boxes else 0}", 
                   (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        return annotated_frame, results[0]
    
    def run_realtime(self, duration=None):
        """Run real-time segmentation"""
        if not camera_available:
            print("❌ Camera not available. Please check camera connection.")
            return
        
        try:
            self.initialize_camera()
            self.running = True
            
            print("\\n🎥 Starting real-time segmentation...")
            print("Press 'q' to quit, 's' to save screenshot, 'r' to toggle recording")
            print("=" * 60)
            
            start_time = time.time()
            
            while self.running:
                ret, frame = self.cap.read()
                
                if not ret:
                    print("Failed to capture frame")
                    break
                
                # Process frame
                annotated_frame, results = self.process_frame(frame)
                
                # Update FPS
                self.update_fps()
                
                # Record if enabled
                if self.record and self.out:
                    self.out.write(annotated_frame)
                
                # Display frame
                cv2.imshow('Real-Time Segmentation', annotated_frame)
                
                # Handle key presses
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    print("\\nQuitting...")
                    break
                elif key == ord('s'):
                    # Save screenshot
                    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                    screenshot_name = f"screenshot_{timestamp}.jpg"
                    cv2.imwrite(screenshot_name, annotated_frame)
                    print(f"Screenshot saved: {screenshot_name}")
                elif key == ord('r'):
                    # Toggle recording
                    if not self.record:
                        self.record = True
                        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                        self.output_file = f"camera_recording_{timestamp}.mp4"
                        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                        actual_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                        actual_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                        self.out = cv2.VideoWriter(self.output_file, fourcc, 20.0, (actual_width, actual_height))
                        print(f"Started recording: {self.output_file}")
                    else:
                        self.record = False
                        if self.out:
                            self.out.release()
                            self.out = None
                        print("Stopped recording")
                
                # Check duration limit
                if duration and (time.time() - start_time) > duration:
                    print(f"\\nReached duration limit of {duration} seconds")
                    break
            
        except KeyboardInterrupt:
            print("\\nInterrupted by user")
        except Exception as e:
            print(f"\\nError: {e}")
        finally:
            self.running = False
            self.cleanup()
            
            # Print final statistics
            if self.processing_times:
                avg_processing_time = np.mean(self.processing_times)
                avg_fps = 1.0 / avg_processing_time
                print(f"\\n📊 Session Statistics:")
                print(f"  Average processing FPS: {avg_fps:.1f}")
                print(f"  Average processing time: {avg_processing_time:.3f}s")
                print(f"  Total frames processed: {len(self.processing_times)}")

# Create segmentation instance
segmenter = RealTimeSegmentation(
    model=model,
    camera_index=CAMERA_INDEX,
    conf_threshold=CONFIDENCE_THRESHOLD,
    frame_width=FRAME_WIDTH,
    frame_height=FRAME_HEIGHT,
    record=RECORD_OUTPUT,
    output_file=OUTPUT_FILENAME if RECORD_OUTPUT else None
)


In [ ]:
# Run real-time segmentation
# Uncomment and run this cell to start real-time segmentation

print("🚀 Ready to start real-time camera segmentation!")
print("\\nTo start, uncomment and run the following line:")
print("segmenter.run_realtime()")
print("\\nOr run for a specific duration (in seconds):")
print("segmenter.run_realtime(duration=30)")

# Uncomment one of these lines to start:
# segmenter.run_realtime()  # Run indefinitely until 'q' is pressed
# segmenter.run_realtime(duration=30)  # Run for 30 seconds


In [ ]:
# Advanced features and utilities

def capture_single_frame():
    """Capture and process a single frame for testing"""
    if not camera_available:
        print("❌ Camera not available")
        return None
    
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print("❌ Cannot open camera")
        return None
    
    try:
        ret, frame = cap.read()
        if ret:
            # Process with YOLO
            results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
            annotated_frame = results[0].plot()
            
            # Display using matplotlib
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            
            # Original frame
            axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            axes[0].set_title('Original Camera Feed')
            axes[0].axis('off')
            
            # Segmented frame
            axes[1].imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
            axes[1].set_title('Segmented Output')
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            # Print detection info
            if results[0].boxes is not None:
                print(f"\\nDetected {len(results[0].boxes)} objects:")
                for i, (cls, conf) in enumerate(zip(results[0].boxes.cls, results[0].boxes.conf)):
                    class_name = results[0].names[int(cls)]
                    print(f"  {i+1}. {class_name}: {conf:.3f}")
            else:
                print("No objects detected")
                
            return annotated_frame
        else:
            print("❌ Failed to capture frame")
            return None
    finally:
        cap.release()

def benchmark_performance(duration=10):
    """Benchmark segmentation performance"""
    if not camera_available:
        print("❌ Camera not available")
        return
    
    print(f"🔄 Running {duration}-second performance benchmark...")
    
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print("❌ Cannot open camera")
        return
    
    try:
        processing_times = []
        frame_count = 0
        start_time = time.time()
        
        while (time.time() - start_time) < duration:
            ret, frame = cap.read()
            if not ret:
                continue
            
            # Time the processing
            process_start = time.time()
            results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
            annotated_frame = results[0].plot()
            process_time = time.time() - process_start
            
            processing_times.append(process_time)
            frame_count += 1
        
        # Calculate statistics
        avg_time = np.mean(processing_times)
        min_time = np.min(processing_times)
        max_time = np.max(processing_times)
        avg_fps = 1.0 / avg_time
        
        print(f"\\n📊 Benchmark Results ({duration}s):")
        print(f"  Frames processed: {frame_count}")
        print(f"  Average processing time: {avg_time:.3f}s")
        print(f"  Min processing time: {min_time:.3f}s")
        print(f"  Max processing time: {max_time:.3f}s")
        print(f"  Average FPS: {avg_fps:.1f}")
        print(f"  Theoretical max FPS: {1.0/min_time:.1f}")
        
    finally:
        cap.release()

# Test functions
print("🔧 Utility functions available:")
print("  capture_single_frame() - Capture and display one frame")
print("  benchmark_performance(duration=10) - Run performance benchmark")
print("\\nExample usage:")
print("  capture_single_frame()")
print("  benchmark_performance(duration=5)")


In [ ]:
# Tips and troubleshooting

print("💡 TIPS FOR OPTIMAL PERFORMANCE:")
print("=" * 50)
print("1. 🚀 SPEED OPTIMIZATION:")
print("   - Use yolo11n-seg for fastest processing")
print("   - Reduce frame resolution (e.g., 320x240 for very fast)")
print("   - Increase confidence threshold to reduce processing")
print("   - Close other applications to free up resources")
print()
print("2. 🎯 ACCURACY OPTIMIZATION:")
print("   - Use yolo11s-seg or yolo11m-seg for better accuracy")
print("   - Lower confidence threshold (0.3-0.4)")
print("   - Ensure good lighting conditions")
print("   - Keep camera stable")
print()
print("3. 🔧 TROUBLESHOOTING:")
print("   - If camera not detected: Try different CAMERA_INDEX values (0, 1, 2...)")
print("   - If low FPS: Reduce resolution or use smaller model")
print("   - If high CPU usage: Enable GPU acceleration if available")
print("   - If recording fails: Check disk space and write permissions")
print()
print("4. ⌨️  KEYBOARD CONTROLS:")
print("   - 'q': Quit the application")
print("   - 's': Save screenshot of current frame")
print("   - 'r': Toggle recording on/off")
print()
print("5. 📊 PERFORMANCE MONITORING:")
print("   - Green text overlay shows real-time FPS")
print("   - Processing FPS indicates model performance")
print("   - Object count shows detection results")
print()
print("6. 🎥 RECORDING NOTES:")
print("   - Videos saved as MP4 format")
print("   - Filename includes timestamp")
print("   - Recording can be toggled during runtime")
print("   - Check output file after stopping")

print("\\n" + "=" * 50)
print("🚀 READY TO START REAL-TIME SEGMENTATION!")
print("Run the previous cell to begin camera segmentation.")
print("=" * 50)
